# TCN Receptor-Level Holdout Experiments
This notebook tracks commands and results for rerunning TCN experiments under **receptor-level holdout** (within-family generalization), to complement the existing cross-family results.

## Motivation
With the paper reframed to lead with within-family generalization as the primary evaluation, we need:
1. Handcrafted features (scalar, TICA, combined) under receptor-level holdout → expected to still fail, confirming failure is not split-dependent
2. Oracle/sanity check features under receptor-level holdout → upper bound under easier evaluation setting


## Summary of Results (filled in based on below completed experiments)
| Feature | Model | Dim | AUROC (receptor) | AUROC (family) | Behaviour |
|---------|-------|-----|-----------------|----------------|-----------|
Scalar | TCN | 3 | 0.682 | 0.500 | Degenerate 
TICA projections | TCN | 5 | 0.782 | 0.500 | Degenerate 
Combined | TCN | 58 | 0.564 | 0.543 | Degenerate 
ESM-2 | LR | 1280 | 0.688 | 0.541 | Near-chance 
ESM-2 | RF | 1280 | 0.560 | 0.513 | Near-chance 
ESM-2 | MLP | 1280 | 0.521 | 0.529 | Near-chance 
ESMDance | LR | 480 | 0.554 | 0.494 | Near-chance 
ESMDance | RF | 480 | 0.577 | 0.523 | Near-chance 
ESMDance | MLP | 480 | 0.619 | 0.500 | Degenerate 
ESM-IF (mean-pooled) | LR | 512 | -- | 0.541 | Near-chance 
ESM-IF (mean-pooled) | RF | 512 | -- | 0.513 | Near-chance 
ESM-IF (mean-pooled) | MLP | 512 | -- | 0.529 | Near-chance 
Oracle (10\%) | TCN | 7 | 0.836 | 0.783 | Real signal 
Oracle (50\%) | TCN | 7 | 0.862 | 0.811 | Real signal 
Oracle (90\%) | TCN | 7 | 0.974 | 0.933 | Real signal 


## Step 1: Generate Receptor-Level Split Files
Uses `within-protein` split mode from `generate_split_files.py`.
This holds out 20% of receptors, allowing the same family to appear in both train and test.

In [10]:
# # Scalar + combined features at 50%
# !python src/data/generate_split_files.py within-protein \
#     --metadata_csv data/metadata/metadata.csv \
#     --feature_type combined \
#     --trajectory_pct 50 \
#     --output_dir data/metadata/splits

# # Sanity check features at 50%
# !python src/data/generate_split_files.py within-protein \
#     --metadata_csv data/metadata/metadata.csv \
#     --feature_type sanity_check \
#     --trajectory_pct 50 \
#     --output_dir data/metadata/splits

# # Sanity check features at 90%
# !python src/data/generate_split_files.py within-protein \
#     --metadata_csv data/metadata/metadata.csv \
#     --feature_type sanity_check \
#     --trajectory_pct 90 \
#     --output_dir data/metadata/splits

## Step 2: Train TCN Models Under Receptor-Level Holdout
Run 3 seeds each for variance estimation, consistent with trajectory sweep methodology.

In [ ]:
# --- Handcrafted features (scalar) at 50% ---
# Expected: still chance-level, confirming failure is not split-dependent

for seed in [0, 1, 2]:
    !python train_tcn.py \
        --train_csv ../../data/metadata/splits/protein_level_train.csv \
        --test_csv ../../data/metadata/splits/protein_level_test.csv \
        --npy_dir ../../../gcs_mount/data/features_50pct/scalar \
        --batch_size 1 --num_workers 0 \
        --patience 50 --epochs 200 \
        --output_dir ../../results/tcn_scalar_50pct_receptor

Using device: cpu
Output directory: ../../results/tcn_scalar_50pct_receptor/tcn_20260312_150355

Loading datasets...
Loaded 203 samples
  Multi: 62.0 (30.5%)
  Single: 141 (69.5%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 3 features from data

Initializing TCN with 3 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        13,676
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=10.1845, Bal Acc=0.511, AUROC=0.549, AP=0.339
  Test:  Loss=5.3012, Bal Acc=0.515, AUROC=0.418, AP=0.186
   New best AUROC: 0.418
Epoch 2/200
  Train: Loss=2.2483, Bal Acc=0.486, AUROC=0.439, AP=0.267
  Test:  Loss=0.7758, Bal Acc=0.500, AUROC=0.748, AP=0.503
   New best AUROC: 0.748
Epoch 3/200
  Train: Loss=1.8082, Bal Acc=0.500, AUROC=0.555, AP=0.348
  Test:  Loss=0.7742, B

In [ ]:
# --- Handcrafted features (TICA) at 50% ---
# Expected: still chance-level, confirming failure is not split-dependent

for seed in [0, 1, 2]:
    !python train_tcn.py \
        --train_csv ../../data/metadata/splits/protein_level_tica_train.csv \
        --test_csv ../../data/metadata/splits/protein_level_test.csv \
        --npy_dir ../../../gcs_mount/data/features_50pct/tica/projections \
        --batch_size 1 --num_workers 0 \
        --patience 50 --epochs 200 \
        --output_dir ../../results/tcn_tica_50pct_receptor

Using device: cpu
Output directory: ../../results/tcn_tica_50pct_receptor/tcn_20260312_154322

Loading datasets...
Loaded 200 samples
  Multi: 62.0 (31.0%)
  Single: 138 (69.0%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 5 features from data

Initializing TCN with 5 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        13,876
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=1.7045, Bal Acc=0.500, AUROC=0.393, AP=0.264
  Test:  Loss=0.8035, Bal Acc=0.500, AUROC=0.660, AP=0.448
   New best AUROC: 0.660
Epoch 2/200
  Train: Loss=1.1784, Bal Acc=0.500, AUROC=0.537, AP=0.328
  Test:  Loss=0.7345, Bal Acc=0.500, AUROC=0.706, AP=0.552
   New best AUROC: 0.706
Epoch 3/200
  Train: Loss=1.0430, Bal Acc=0.500, AUROC=0.589, AP=0.360
  Test:  Loss=0.5327, Bal 

In [6]:
# --- Handcrafted features (combined) at 50% ---
# Expected: still chance-level, confirming failure is not split-dependent

for seed in [0, 1, 2]:
    !python train_tcn.py \
        --train_csv ../../data/metadata/splits/protein_level_train.csv \
        --test_csv ../../data/metadata/splits/protein_level_test.csv \
        --npy_dir ../../../gcs_mount/data/processed/combined_features/combined_50pct \
        --batch_size 1 --num_workers 0 \
        --patience 50 --epochs 200 \
        --output_dir ../../results/tcn_combined_50pct_receptor


Using device: cpu
Output directory: ../../results/tcn_combined_50pct_receptor/tcn_20260312_120523

Loading datasets...
Loaded 203 samples
  Multi: 62.0 (30.5%)
  Single: 141 (69.5%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 58 features from data

Initializing TCN with 58 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        19,176
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
/opt/conda/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/opt/conda/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memo

NameError: name 'done' is not defined

In [ ]:
# --- Oracle/sanity check features at 50% ---
# Upper bound under receptor-level holdout; expect higher than 0.831 (cross-family)

for seed in [0, 1, 2]:
    !python train_tcn.py \
        --train_csv ../../data/metadata/splits/protein_level_train.csv \
        --test_csv ../../data/metadata/splits/protein_level_test.csv \
        --npy_dir ../../data/features_50pct/oracle_features \
        --batch_size 1 --num_workers 0 \
        --patience 50 --epochs 200 \
        --output_dir ../../results/tcn_oracle_50pct_receptor

Using device: cpu
Output directory: results/tcn_oracle_90pct_receptor/tcn_20260312_123236

Loading datasets...
Loaded 203 samples
  Multi: 62.0 (30.5%)
  Single: 141 (69.5%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 7 features from data

Initializing TCN with 7 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        14,076
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=1.7736, Bal Acc=0.486, AUROC=0.503, AP=0.319
  Test:  Loss=1.0830, Bal Acc=0.500, AUROC=0.657, AP=0.459
   New best AUROC: 0.657
Epoch 2/200
  Train: Loss=1.2580, Bal Acc=0.500, AUROC=0.484, AP=0.295
  Test:  Loss=0.6862, Bal Acc=0.500, AUROC=0.797, AP=0.546
   New best AUROC: 0.797
Epoch 3/200
  Train: Loss=1.1979, Bal Acc=0.500, AUROC=0.540, AP=0.343
  Test:  Loss=0.8451, Bal Acc=

NameError: name 'done' is not defined

In [9]:
# --- Oracle/sanity check features at 90% ---
# Upper bound at 50%; expect close to cross-family result of 0.95

for seed in [0, 1, 2]:
    !python train_tcn.py \
        --train_csv ../../data/metadata/splits/protein_level_train.csv \
        --test_csv ../../data/metadata/splits/protein_level_test.csv \
        --npy_dir ../../data/processed_v4/features_90pct/sanity_check_features \
        --batch_size 1 --num_workers 0 \
        --patience 50 --epochs 200 \
        --output_dir ../../results/tcn_oracle_90pct_receptor


Using device: cpu
Output directory: ../../results/tcn_oracle_90pct_receptor/tcn_20260312_143839

Loading datasets...
Loaded 203 samples
  Multi: 62.0 (30.5%)
  Single: 141 (69.5%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 7 features from data

Initializing TCN with 7 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        14,076
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=1.7125, Bal Acc=0.490, AUROC=0.494, AP=0.293
  Test:  Loss=0.7735, Bal Acc=0.500, AUROC=0.709, AP=0.477
   New best AUROC: 0.709
Epoch 2/200
  Train: Loss=1.2216, Bal Acc=0.500, AUROC=0.576, AP=0.357
  Test:  Loss=0.5264, Bal Acc=0.500, AUROC=0.758, AP=0.522
   New best AUROC: 0.758
Epoch 3/200
  Train: Loss=1.1786, Bal Acc=0.500, AUROC=0.554, AP=0.344
  Test:  Loss=0.5509, Ba

In [13]:
# --- Oracle/sanity check features at 10% ---
# Upper bound at 10%

for seed in [0, 1, 2]:
    !python train_tcn.py \
        --train_csv ../../data/metadata/splits/protein_level_train.csv \
        --test_csv ../../data/metadata/splits/protein_level_test.csv \
        --npy_dir ../../data/processed_v4/features_10pct/sanity_check_features \
        --batch_size 1 --num_workers 0 \
        --patience 50 --epochs 200 \
        --output_dir ../../results/tcn_oracle_10pct_receptor

Using device: cpu
Output directory: ../../results/tcn_oracle_10pct_receptor/tcn_20260312_153323

Loading datasets...
Loaded 203 samples
  Multi: 62.0 (30.5%)
  Single: 141 (69.5%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 7 features from data

Initializing TCN with 7 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        14,076
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=1.8451, Bal Acc=0.497, AUROC=0.395, AP=0.280
  Test:  Loss=0.8090, Bal Acc=0.500, AUROC=0.448, AP=0.305
   New best AUROC: 0.448
Epoch 2/200
  Train: Loss=1.3810, Bal Acc=0.500, AUROC=0.393, AP=0.260
  Test:  Loss=0.6802, Bal Acc=0.500, AUROC=0.621, AP=0.343
   New best AUROC: 0.621
Epoch 3/200
  Train: Loss=1.2222, Bal Acc=0.508, AUROC=0.558, AP=0.395
  Test:  Loss=1.4762, Ba

In [34]:
# --- Oracle/sanity check features at 20,30,40,60,70,80,100% ---

for pct in [20,30,40,60,70,80,100]:
    for seed in [0, 1, 2]:
        !python train_tcn.py \
            --train_csv ../../data/metadata/splits/protein_level_train.csv \
            --test_csv ../../data/metadata/splits/protein_level_test.csv \
            --npy_dir ../../data/processed_v4/features_{pct}pct/sanity_check_features \
            --batch_size 1 --num_workers 0 \
            --patience 50 --epochs 200 \
            --output_dir ../../results/tcn_oracle_{pct}pct_receptor

Using device: cpu
Output directory: ../../results/tcn_oracle_20pct_receptor/tcn_20260313_004158

Loading datasets...
Loaded 203 samples
  Multi: 62.0 (30.5%)
  Single: 141 (69.5%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 7 features from data

Initializing TCN with 7 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        14,076
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=1.9129, Bal Acc=0.485, AUROC=0.492, AP=0.295
  Test:  Loss=1.0259, Bal Acc=0.500, AUROC=0.386, AP=0.187
   New best AUROC: 0.386
Epoch 2/200
  Train: Loss=1.3290, Bal Acc=0.496, AUROC=0.533, AP=0.348
  Test:  Loss=1.1082, Bal Acc=0.500, AUROC=0.399, AP=0.194
   New best AUROC: 0.399
Epoch 3/200
  Train: Loss=1.2457, Bal Acc=0.500, AUROC=0.526, AP=0.349
  Test:  Loss=0.5889, Ba

## Step 3: Collect Results
Aggregates AUROC and balanced accuracy across seeds for each experiment.
Edit `result_dirs` to match actual output directory names after training.

In [21]:
import os
import json
import glob
import numpy as np
import pandas as pd

def collect_results(base_dir, label):
    """
    Collect AUROC, balanced accuracy, AP, F1, and confusion matrix across seed subdirectories.
    Looks for metrics.json or test_predictions.csv in each seed folder.
    Returns a dict with mean/std suitable for pasting into the results table.
    """
    from sklearn.metrics import (
        roc_auc_score, balanced_accuracy_score,
        average_precision_score, f1_score, confusion_matrix
    )

    aurocs, bal_accs, aps, f1s = [], [], [], []
    cms = []
    behaviours = []

    seed_dirs = sorted(glob.glob(os.path.join(base_dir, 'seed_*')))
    if not seed_dirs:
        seed_dirs = sorted(glob.glob(os.path.join(base_dir, 'tcn_*')))

    for seed_dir in seed_dirs:
        print(f'found {base_dir} / {seed_dir} for {label}')
        metrics_path = os.path.join(seed_dir, 'metrics.json')

        if os.path.exists(metrics_path):
            with open(metrics_path) as f:
                m = json.load(f)
            aurocs.append(m.get('test_auroc', m.get('auroc')))
            bal_accs.append(m.get('test_balanced_accuracy', m.get('balanced_accuracy')))
            aps.append(m.get('test_ap', m.get('ap')))
            f1s.append(m.get('test_f1', m.get('f1')))
            # metrics.json may not have CM — skip if absent
            cm = m.get('confusion_matrix')
            cms.append(np.array(cm) if cm is not None else None)
            # determine degeneracy from pred distribution if available
            unique_preds = m.get('unique_pred_labels')
            is_degenerate = (unique_preds is not None and len(unique_preds) == 1)

        else:
            pred_path = os.path.join(seed_dir, 'test_predictions.csv')
            if os.path.exists(pred_path):
                preds = pd.read_csv(pred_path)
                print(f'  columns: {list(preds.columns)}')
                y_true = preds['true_label']
                y_prob = preds['pred_prob']
                y_pred = preds['pred_label']

                aurocs.append(roc_auc_score(y_true, y_prob))
                bal_accs.append(balanced_accuracy_score(y_true, y_pred))
                aps.append(average_precision_score(y_true, y_prob))
                f1s.append(f1_score(y_true, y_pred, zero_division=0))
                cms.append(confusion_matrix(y_true, y_pred))
                is_degenerate = (len(y_pred.unique()) == 1)
            else:
                print(f'  WARNING: no metrics found in {seed_dir}')
                continue

        # Behaviour classification — degeneracy takes priority over AUROC threshold
        if is_degenerate:
            behaviours.append('Degenerate')
        elif aurocs and aurocs[-1] is not None:
            a = aurocs[-1]
            if a > 0.6:
                behaviours.append('Real signal')
            elif abs(a - 0.5) < 0.05:
                behaviours.append('Chance')
            else:
                behaviours.append('Near-chance')
        else:
            behaviours.append('Unknown')

    if not aurocs:
        print(f'No results found in {base_dir}')
        return None

    # Aggregate confusion matrices
    valid_cms = [cm for cm in cms if cm is not None]
    cm_mean = np.mean(valid_cms, axis=0).round(1) if valid_cms else None
    cm_std  = np.std(valid_cms,  axis=0).round(1) if valid_cms else None

    result = {
        'label':        label,
        'n_seeds':      len(aurocs),
        'auroc_mean':   np.mean(aurocs),
        'auroc_std':    np.std(aurocs),
        'bal_acc_mean': np.mean(bal_accs) if bal_accs else None,
        'bal_acc_std':  np.std(bal_accs)  if bal_accs else None,
        'ap_mean':      np.mean(aps)      if aps      else None,
        'ap_std':       np.std(aps)       if aps      else None,
        'f1_mean':      np.mean(f1s)      if f1s      else None,
        'f1_std':       np.std(f1s)       if f1s      else None,
        'cm_mean':      cm_mean,
        'cm_std':       cm_std,
        'behaviour':    behaviours[-1]    if behaviours else 'Unknown',
        'raw_aurocs':   aurocs,
    }

    # Pretty print
    print(f"{label}:")
    print(f"  AUROC:    {result['auroc_mean']:.3f} ± {result['auroc_std']:.3f}")
    print(f"  Bal Acc:  {result['bal_acc_mean']:.3f} ± {result['bal_acc_std']:.3f}" if bal_accs else "  Bal Acc: N/A")
    print(f"  AP:       {result['ap_mean']:.3f} ± {result['ap_std']:.3f}"           if aps      else "  AP:      N/A")
    print(f"  F1:       {result['f1_mean']:.3f} ± {result['f1_std']:.3f}"           if f1s      else "  F1:      N/A")
    print(f"  Behaviour: {result['behaviour']} (n={result['n_seeds']})")
    if cm_mean is not None:
        print(f"  CM (mean): TN={cm_mean[0,0]}, FP={cm_mean[0,1]}, FN={cm_mean[1,0]}, TP={cm_mean[1,1]}")

    return result


# --- Edit these paths to match actual output dirs ---
experiments = [
    ('../../results/tcn_scalar_50pct_receptor',         'Scalar (50%)',                   'receptor'),
    ('../../results/tcn_tica_50pct_receptor',           'TICA (50%)',                     'receptor'),
    ('../../results/tcn_combined_50pct_receptor',       'Handcrafted (combined, 50%)',    'receptor'),
    ('../../results/tcn_oracle_10pct_receptor',         'Oracle (10%)',                   'receptor'),
    ('../../results/tcn_oracle_50pct_receptor',         'Oracle (50%)',                   'receptor'),
    ('../../results/tcn_oracle_90pct_receptor',         'Oracle (90%)',                   'receptor'),
]

all_results = []
for base_dir, label, split in experiments:
    r = collect_results(base_dir, label)
    if r:
        r['split'] = split
        all_results.append(r)

results_df = pd.DataFrame(all_results)
results_df

found ../../results/tcn_scalar_50pct_receptor / ../../results/tcn_scalar_50pct_receptor/tcn_20260312_150355 for Scalar (50%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_scalar_50pct_receptor / ../../results/tcn_scalar_50pct_receptor/tcn_20260312_151413 for Scalar (50%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_scalar_50pct_receptor / ../../results/tcn_scalar_50pct_receptor/tcn_20260312_151825 for Scalar (50%)
  columns: ['true_label', 'pred_prob', 'pred_label']
Scalar (50%):
  AUROC:    0.682 ± 0.151
  Bal Acc:  0.500 ± 0.000
  AP:       0.377 ± 0.158
  F1:       0.115 ± 0.163
  Behaviour: Degenerate (n=3)
  CM (mean): TN=22.7, FP=11.3, FN=6.0, TP=3.0
found ../../results/tcn_tica_50pct_receptor / ../../results/tcn_tica_50pct_receptor/tcn_20260312_154322 for TICA (50%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_tica_50pct_receptor / ../../results/tcn_tica_50pct_receptor/tcn_20260312_154

,label,n_seeds,auroc_mean,auroc_std,bal_acc_mean,bal_acc_std,ap_mean,ap_std,f1_mean,f1_std,cm_mean,cm_std,behaviour,raw_aurocs,split
0,Scalar (50%),3,0.681917,0.150808,0.500000,0.000000,0.377065,0.157686,0.115385,0.163178,"[[22.7, 11.3], [6.0, 3.0]]","[[16.0, 16.0], [4.2, 4.2]]",Degenerate,"[0.869281045751634, 0.6764705882352942, 0.5]",receptor
1,TICA (50%),3,0.782135,0.022218,0.581699,0.115540,0.641203,0.024386,0.190476,0.269374,"[[32.0, 2.0], [7.0, 2.0]]","[[2.8, 2.8], [2.8, 2.8]]",Degenerate,"[0.7516339869281046, 0.7908496732026143, 0.803...",receptor
2,"Handcrafted (combined, 50%)",3,0.564270,0.090892,0.500000,0.000000,0.253360,0.062306,0.115385,0.163178,"[[22.7, 11.3], [6.0, 3.0]]","[[16.0, 16.0], [4.2, 4.2]]",Degenerate,"[0.6928104575163399, 0.5, 0.5]",receptor
3,Oracle (10%),3,0.836057,0.021609,0.689542,0.012007,0.519847,0.061272,0.500835,0.020472,"[[28.0, 6.0], [4.0, 5.0]]","[[0.8, 0.8], [0.0, 0.0]]",Real signal,"[0.8055555555555556, 0.8496732026143791, 0.852...",receptor
4,Oracle (50%),3,0.862200,0.005392,0.708061,0.107846,0.571595,0.042170,0.460041,0.184505,"[[28.0, 6.0], [3.7, 5.3]]","[[4.5, 4.5], [3.1, 3.1]]",Real signal,"[0.861111111111111, 0.869281045751634, 0.85620...",receptor
5,Oracle (90%),3,0.973856,0.004622,0.863290,0.016893,0.924197,0.005009,0.758842,0.016852,"[[31.0, 3.0], [1.7, 7.3]]","[[0.8, 0.8], [0.5, 0.5]]",Real signal,"[0.9673202614379085, 0.9771241830065359, 0.977...",receptor


In [35]:
oracle_experiments = [
    ('../../results/tcn_oracle_10pct_receptor',         'Oracle (10%)',                   'receptor'),
    ('../../results/tcn_oracle_20pct_receptor',         'Oracle (20%)',                   'receptor'),
    ('../../results/tcn_oracle_30pct_receptor',         'Oracle (30%)',                   'receptor'),
    ('../../results/tcn_oracle_40pct_receptor',         'Oracle (40%)',                   'receptor'),
    ('../../results/tcn_oracle_50pct_receptor',         'Oracle (50%)',                   'receptor'),
    ('../../results/tcn_oracle_60pct_receptor',         'Oracle (60%)',                   'receptor'),
    ('../../results/tcn_oracle_70pct_receptor',         'Oracle (70%)',                   'receptor'),
    ('../../results/tcn_oracle_80pct_receptor',         'Oracle (80%)',                   'receptor'),
    ('../../results/tcn_oracle_90pct_receptor',         'Oracle (90%)',                   'receptor'),
    ('../../results/tcn_oracle_100pct_receptor',         'Oracle (100%)',                 'receptor'),
]

all_oracle_results = []
for base_dir, label, split in oracle_experiments:
    r = collect_results(base_dir, label)
    if r:
        r['split'] = split
        all_oracle_results.append(r)

oracle_results_df = pd.DataFrame(all_oracle_results)
oracle_results_df

found ../../results/tcn_oracle_10pct_receptor / ../../results/tcn_oracle_10pct_receptor/tcn_20260312_153323 for Oracle (10%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_oracle_10pct_receptor / ../../results/tcn_oracle_10pct_receptor/tcn_20260312_153613 for Oracle (10%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_oracle_10pct_receptor / ../../results/tcn_oracle_10pct_receptor/tcn_20260312_153819 for Oracle (10%)
  columns: ['true_label', 'pred_prob', 'pred_label']
Oracle (10%):
  AUROC:    0.836 ± 0.022
  Bal Acc:  0.690 ± 0.012
  AP:       0.520 ± 0.061
  F1:       0.501 ± 0.020
  Behaviour: Real signal (n=3)
  CM (mean): TN=28.0, FP=6.0, FN=4.0, TP=5.0
found ../../results/tcn_oracle_20pct_receptor / ../../results/tcn_oracle_20pct_receptor/tcn_20260313_004158 for Oracle (20%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_oracle_20pct_receptor / ../../results/tcn_oracle_20pct_receptor/tcn_20

,label,n_seeds,auroc_mean,auroc_std,bal_acc_mean,bal_acc_std,ap_mean,ap_std,f1_mean,f1_std,cm_mean,cm_std,behaviour,raw_aurocs,split
0,Oracle (10%),3,0.836057,0.021609,0.689542,0.012007,0.519847,0.061272,0.500835,0.020472,"[[28.0, 6.0], [4.0, 5.0]]","[[0.8, 0.8], [0.0, 0.0]]",Real signal,"[0.8055555555555556, 0.8496732026143791, 0.852...",receptor
1,Oracle (20%),3,0.816993,0.027080,0.601852,0.095243,0.518553,0.108622,0.357260,0.164248,"[[28.3, 5.7], [5.7, 3.3]]","[[1.2, 1.2], [1.7, 1.7]]",Real signal,"[0.781045751633987, 0.8464052287581699, 0.8235...",receptor
2,Oracle (30%),3,0.837146,0.015578,0.814270,0.030257,0.504907,0.008461,0.646904,0.036274,"[[27.7, 6.3], [1.7, 7.3]]","[[0.5, 0.5], [0.5, 0.5]]",Real signal,"[0.8333333333333333, 0.857843137254902, 0.8202...",receptor
3,Oracle (40%),3,0.851852,0.012324,0.637800,0.016893,0.557548,0.031131,0.425199,0.022460,"[[27.0, 7.0], [4.7, 4.3]]","[[0.8, 0.8], [0.5, 0.5]]",Real signal,"[0.8431372549019607, 0.8692810457516339, 0.843...",receptor
4,Oracle (50%),3,0.862200,0.005392,0.708061,0.107846,0.571595,0.042170,0.460041,0.184505,"[[28.0, 6.0], [3.7, 5.3]]","[[4.5, 4.5], [3.1, 3.1]]",Real signal,"[0.861111111111111, 0.869281045751634, 0.85620...",receptor
5,Oracle (60%),3,0.887255,0.027762,0.818083,0.059025,0.685641,0.062323,0.646789,0.091347,"[[26.7, 7.3], [1.3, 7.7]]","[[2.5, 2.5], [0.5, 0.5]]",Real signal,"[0.8480392156862744, 0.9084967320261438, 0.905...",receptor
6,Oracle (70%),3,0.870915,0.037570,0.777233,0.101468,0.667775,0.129247,0.594457,0.123185,"[[27.7, 6.3], [2.3, 6.7]]","[[0.5, 0.5], [1.9, 1.9]]",Real signal,"[0.8316993464052288, 0.9215686274509803, 0.859...",receptor
7,Oracle (80%),3,0.936275,0.024052,0.848584,0.082957,0.705179,0.063489,0.710916,0.087159,"[[30.0, 4.0], [1.7, 7.3]]","[[0.8, 0.8], [1.7, 1.7]]",Real signal,"[0.9035947712418301, 0.9607843137254902, 0.944...",receptor
8,Oracle (90%),3,0.973856,0.004622,0.863290,0.016893,0.924197,0.005009,0.758842,0.016852,"[[31.0, 3.0], [1.7, 7.3]]","[[0.8, 0.8], [0.5, 0.5]]",Real signal,"[0.9673202614379085, 0.9771241830065359, 0.977...",receptor
9,Oracle (100%),3,0.977124,0.013865,0.922658,0.013162,0.915552,0.051985,0.803429,0.039022,"[[30.0, 4.0], [0.3, 8.7]]","[[1.6, 1.6], [0.5, 0.5]]",Real signal,"[0.957516339869281, 0.9869281045751633, 0.9869...",receptor


In [22]:
family_experiments = [
    ('../../results/tcn_sanity_check_10pct',            'Oracle (10%)',                   'family'),
    ('../../results/tcn_sanity_check_50pct',            'Oracle (50%)',                   'family'),
    ('../../results/tcn_sanity_check_90pct',            'Oracle (90%)',                   'family'),
]

all_family_results = []
for base_dir, label, split in family_experiments:
    r = collect_results(base_dir, label)
    if r:
        r['split'] = split
        all_family_results.append(r)

family_results_df = pd.DataFrame(all_family_results)
family_results_df

found ../../../gcs_mount/results/tcn_combined_50pct / ../../../gcs_mount/results/tcn_combined_50pct/tcn_20260228_152756 for Handcrafted (combined, 50%)
  columns: ['true_label', 'pred_prob', 'pred_label']
Handcrafted (combined, 50%):
  AUROC:    0.543 ± 0.000
  Bal Acc:  0.500 ± 0.000
  AP:       0.425 ± 0.000
  F1:       0.000 ± 0.000
  Behaviour: Degenerate (n=1)
  CM (mean): TN=35.0, FP=0.0, FN=22.0, TP=0.0
found ../../results/tcn_sanity_check_10pct / ../../results/tcn_sanity_check_10pct/tcn_20260311_032034 for Oracle (10%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_sanity_check_10pct / ../../results/tcn_sanity_check_10pct/tcn_20260311_032501 for Oracle (10%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_sanity_check_10pct / ../../results/tcn_sanity_check_10pct/tcn_20260311_032806 for Oracle (10%)
  columns: ['true_label', 'pred_prob', 'pred_label']
found ../../results/tcn_sanity_check_10pct / ../../results/tcn_sanity_

,label,n_seeds,auroc_mean,auroc_std,bal_acc_mean,bal_acc_std,ap_mean,ap_std,f1_mean,f1_std,cm_mean,cm_std,behaviour,raw_aurocs,split
0,"Handcrafted (combined, 50%)",1,0.542857,0.000000,0.500000,0.000000,0.424977,0.000000,0.000000,0.000000,"[[35.0, 0.0], [22.0, 0.0]]","[[0.0, 0.0], [0.0, 0.0]]",Degenerate,[0.5428571428571428],family
1,Oracle (10%),4,0.782468,0.029750,0.554545,0.013721,0.697460,0.024300,0.256287,0.030804,"[[33.2, 1.8], [18.5, 3.5]]","[[0.8, 0.8], [0.5, 0.5]]",Real signal,"[0.8038961038961038, 0.8090909090909091, 0.733...",family
2,Oracle (50%),3,0.811255,0.020461,0.747835,0.033086,0.761445,0.044381,0.678679,0.045186,"[[31.7, 3.3], [9.0, 13.0]]","[[1.2, 1.2], [0.8, 0.8]]",Real signal,"[0.7831168831168832, 0.8311688311688312, 0.819...",family
3,Oracle (90%),4,0.933117,0.014650,0.727922,0.055638,0.912547,0.025040,0.625781,0.096400,"[[34.2, 0.8], [11.5, 10.5]]","[[0.8, 0.8], [2.3, 2.3]]",Real signal,"[0.9558441558441557, 0.9337662337662338, 0.927...",family


## Step 4: Format for Report
Generates LaTeX table rows ready to paste into the paper.
Cross-family numbers are hardcoded from existing results for comparison.

In [23]:
# Cross-family results (hardcoded from existing experiments)
cross_family = {
    'Scalar (TCN, 3-dim)':              {'auroc': 0.500, 'behaviour': 'Degenerate'},
    'TICA projections (TCN, 5-dim)':    {'auroc': 0.500, 'behaviour': 'Degenerate'},
    'Combined (TCN, 58-dim)':           {'auroc': 0.543, 'behaviour': 'Degenerate'},
    'Oracle 10% (TCN, 7-dim)':          {'auroc': 0.783, 'behaviour': 'Real signal'},
    'Oracle 50% (TCN, 7-dim)':          {'auroc': 0.811, 'behaviour': 'Real signal'},
    'Oracle 90% (TCN, 7-dim)':          {'auroc': 0.933, 'behaviour': 'Real signal'},
    'ESMDance LR (480-dim)':            {'auroc': 0.494, 'behaviour': 'Near-chance'},
    'ESMDance RF (480-dim)':            {'auroc': 0.523, 'behaviour': 'Near-chance'},
    'ESMDance MLP (480-dim)':           {'auroc': 0.500, 'behaviour': 'Degenerate'},
    'ESM-IF mean-pooled LR':            {'auroc': 0.541, 'behaviour': 'Near-chance'},
    'ESM-IF mean-pooled RF':            {'auroc': 0.513, 'behaviour': 'Near-chance'},
    'ESM-IF mean-pooled MLP':           {'auroc': 0.529, 'behaviour': 'Near-chance'},
}

# ESM-2,LogisticRegression,1280,0.5409090909090909,0.434697431180753,0.4948051948051948,0.5968253968253968,6,29,4,18
# ESM-2,RandomForest,1280,0.512987012987013,0.42295344159738346,0.5253246753246753,0.551468253968254,32,3,19,3
# ESM-2,MLP,1280,0.5285714285714286,0.4294189444625695,0.49090909090909096,0.6452380952380953,28,7,18,4
# ESMDance,LogisticRegression,480,0.4941558441558441,0.42390253678527784,0.5233766233766234,0.5975396825396826,8,27,4,18
# ESMDance,RandomForest,480,0.5233766233766234,0.4098917286417287,0.4967532467532467,0.5699206349206349,30,5,19,3
# ESMDance,MLP,480,0.5,0.4187125948404144,0.5311688311688312,0.6382539682539683,34,1,20,2

print("=" * 70)
print("RECEPTOR-LEVEL HOLDOUT RESULTS (fill in after experiments complete)")
print("=" * 70)
for r in all_results:
    print(f"  {r['label']}: {r['auroc_mean']:.3f} ± {r['auroc_std']:.3f} [{r['behaviour']}]")

print()
print("=" * 70)
print("LaTeX TABLE ROWS (receptor AUROC | family AUROC)")
print("Fill in TBD values from results_df above")
print("=" * 70)
print()

# Template — fill in receptor_auroc values from results_df after running
rows = [
    # (features, model, dim, receptor_auroc, family_auroc, behaviour)
    ('Scalar',            'TCN',     '3',   'TBD',  '0.500', 'Degenerate'),
    ('TICA projections',  'TCN',     '5',   'TBD',  '0.500', 'Degenerate'),
    ('Combined',          'TCN',     '58',  'TBD',  '0.500', 'Degenerate'),
    ('ESMDance',          'LR',      '480', '--',   '0.494', 'Near-chance'),
    ('ESMDance',          'RF',      '480', '--',   '0.523', 'Near-chance'),
    ('ESMDance',          'MLP',     '480', '--',   '0.500', 'Degenerate'),
    ('ESM-IF (mean-pooled)', 'LR/RF/MLP', '512', '0.770', '0.527', 'Near-chance / Real signal'),
    ('Oracle (50\\%)',    'TCN',     '7',   'TBD',  '0.831', 'Real signal'),
]

print(r'\midrule')
for feat, model, dim, rec_auroc, fam_auroc, beh in rows:
    print(f'{feat} & {model} & {dim} & {rec_auroc} & {fam_auroc} & {beh} \\\\')
print(r'\midrule')
print(r'Oracle (50\%) & TCN & 7 & TBD & 0.831 & Real signal \\')

RECEPTOR-LEVEL HOLDOUT RESULTS (fill in after experiments complete)
  Scalar (50%): 0.682 ± 0.151 [Degenerate]
  TICA (50%): 0.782 ± 0.022 [Degenerate]
  Handcrafted (combined, 50%): 0.564 ± 0.091 [Degenerate]
  Oracle (10%): 0.836 ± 0.022 [Real signal]
  Oracle (50%): 0.862 ± 0.005 [Real signal]
  Oracle (90%): 0.974 ± 0.005 [Real signal]

LaTeX TABLE ROWS (receptor AUROC | family AUROC)
Fill in TBD values from results_df above

\midrule
Scalar & TCN & 3 & TBD & 0.500 & Degenerate \\
TICA projections & TCN & 5 & TBD & 0.500 & Degenerate \\
Combined & TCN & 58 & TBD & 0.500 & Degenerate \\
ESMDance & LR & 480 & -- & 0.494 & Near-chance \\
ESMDance & RF & 480 & -- & 0.523 & Near-chance \\
ESMDance & MLP & 480 & -- & 0.500 & Degenerate \\
ESM-IF (mean-pooled) & LR/RF/MLP & 512 & 0.770 & 0.527 & Near-chance / Real signal \\
Oracle (50\%) & TCN & 7 & TBD & 0.831 & Real signal \\
\midrule
Oracle (50\%) & TCN & 7 & TBD & 0.831 & Real signal \\


In [30]:
# Cross-family results (hardcoded from existing experiments)
# Columns: auroc, avg_precision, balanced_accuracy, cv_auroc, TN, FP, FN, TP
cross_family = {
    'Scalar (TCN, 3-dim)':           {'auroc': 0.500, 'bal_acc': None,  'cv_auroc': None,  'behaviour': 'Degenerate'},
    'TICA projections (TCN, 5-dim)': {'auroc': 0.500, 'bal_acc': None,  'cv_auroc': None,  'behaviour': 'Degenerate'},
    'Combined (TCN, 58-dim)':        {'auroc': 0.543, 'bal_acc': None,  'cv_auroc': None,  'behaviour': 'Degenerate'},
    'Oracle 10% (TCN, 7-dim)':       {'auroc': 0.783, 'bal_acc': None,  'cv_auroc': None,  'behaviour': 'Real signal'},
    'Oracle 50% (TCN, 7-dim)':       {'auroc': 0.811, 'bal_acc': None,  'cv_auroc': None,  'behaviour': 'Real signal'},
    'Oracle 90% (TCN, 7-dim)':       {'auroc': 0.933, 'bal_acc': None,  'cv_auroc': None,  'behaviour': 'Real signal'},
    'ESM-2 LR (1280-dim)':           {'auroc': 0.541, 'bal_acc': 0.495, 'cv_auroc': 0.597, 'behaviour': 'Near-chance', 'TN':  6, 'FP': 29, 'FN':  4, 'TP': 18},
    'ESM-2 RF (1280-dim)':           {'auroc': 0.513, 'bal_acc': 0.525, 'cv_auroc': 0.551, 'behaviour': 'Near-chance', 'TN': 32, 'FP':  3, 'FN': 19, 'TP':  3},
    'ESM-2 MLP (1280-dim)':          {'auroc': 0.529, 'bal_acc': 0.491, 'cv_auroc': 0.645, 'behaviour': 'Near-chance', 'TN': 28, 'FP':  7, 'FN': 18, 'TP':  4},
    'ESMDance LR (480-dim)':         {'auroc': 0.494, 'bal_acc': 0.523, 'cv_auroc': 0.598, 'behaviour': 'Near-chance', 'TN':  8, 'FP': 27, 'FN':  4, 'TP': 18},
    'ESMDance RF (480-dim)':         {'auroc': 0.523, 'bal_acc': 0.497, 'cv_auroc': 0.570, 'behaviour': 'Near-chance', 'TN': 30, 'FP':  5, 'FN': 19, 'TP':  3},
    'ESMDance MLP (480-dim)':        {'auroc': 0.500, 'bal_acc': 0.531, 'cv_auroc': 0.638, 'behaviour': 'Degenerate',  'TN': 34, 'FP':  1, 'FN': 20, 'TP':  2},
}

print("=" * 70)
print("CROSS-FAMILY RESULTS SUMMARY")
print("=" * 70)
print(f"{'Label':<35} {'AUROC':>6} {'BalAcc':>7} {'CV AUROC':>9} {'Behaviour'}")
print("-" * 70)
for label, m in cross_family.items():
    bal = f"{m['bal_acc']:.3f}" if m['bal_acc'] else '  --  '
    cv  = f"{m['cv_auroc']:.3f}" if m['cv_auroc'] else '  --  '
    print(f"{label:<35} {m['auroc']:>6.3f} {bal:>7} {cv:>9}  {m['behaviour']}")

print()
print("=" * 70)
print("RECEPTOR-LEVEL HOLDOUT RESULTS (fill in after experiments complete)")
print("=" * 70)
for r in all_results:
    bal = f"{r['bal_acc_mean']:.3f}" if r['bal_acc_mean'] else '--'
    print(f"  {r['label']}: AUROC {r['auroc_mean']:.3f} ± {r['auroc_std']:.3f} "
          f"| BalAcc {bal} [{r['behaviour']}]")

print()
print("=" * 70)
print("LaTeX TABLE ROWS (receptor AUROC | family AUROC)")
print("Fill in TBD values from results_df above")
print("=" * 70)
print()


# FINAL RECEPTOR-LEVEL RESULTS
#    model         classifier    auroc  balanced_accuracy  TN  FP  FN  TP
# ESMDance LogisticRegression 0.553922           0.612745  19  15   3   6
# ESMDance       RandomForest 0.576797           0.537582  29   5   7   2
# ESMDance                MLP 0.619281           0.500000  34   0   9   0
#    ESM-2 LogisticRegression 0.687908           0.686275  24  10   3   6
#    ESM-2       RandomForest 0.560458           0.537582  29   5   7   2
#    ESM-2                MLP 0.521242           0.467320  28   6   8   1


# Template — fill in receptor_auroc values from results_df after running
# (features, model, dim, receptor_auroc, family_auroc, behaviour)
rows = [
    ('Scalar',               'TCN',     '3',    '0.682', '0.500', 'Degenerate'),
    ('TICA projections',     'TCN',     '5',    '0.782', '0.500', 'Degenerate'),
    ('Combined',             'TCN',     '58',   '0.564', '0.543', 'Degenerate'),
    ('ESM-2',                'LR',        '1280', '0.688',  '0.541', 'Near-chance'),
    ('ESM-2',                'RF',        '1280', '0.560',  '0.513', 'Near-chance'),
    ('ESM-2',                'MLP',       '1280', '0.521',  '0.529', 'Near-chance'),
    ('ESMDance',             'LR',        '480',  '0.554',  '0.494', 'Near-chance'),
    ('ESMDance',             'RF',        '480',  '0.577',  '0.523', 'Near-chance'),
    ('ESMDance',             'MLP',       '480',  '0.619',  '0.500', 'Degenerate'),
    ('ESM-IF (mean-pooled)', 'LR', '512',  '--', '0.541', 'Near-chance'),
    ('ESM-IF (mean-pooled)', 'RF', '512',  '--', '0.513', 'Near-chance'),
    ('ESM-IF (mean-pooled)', 'MLP', '512', '--', '0.529', 'Near-chance'),
    ('Oracle (10\\%)',       'TCN',       '7',    '0.836', '0.783', 'Real signal'),
    ('Oracle (50\\%)',       'TCN',       '7',    '0.862', '0.811', 'Real signal'),
    ('Oracle (90\\%)',       'TCN',       '7',    '0.974', '0.933', 'Real signal'),
]

print(r'\midrule')
for feat, model, dim, rec_auroc, fam_auroc, beh in rows:
    print(f'{feat} & {model} & {dim} & {rec_auroc} & {fam_auroc} & {beh} \\\\')
print(r'\midrule')

CROSS-FAMILY RESULTS SUMMARY
Label                                AUROC  BalAcc  CV AUROC Behaviour
----------------------------------------------------------------------
Scalar (TCN, 3-dim)                  0.500    --        --    Degenerate
TICA projections (TCN, 5-dim)        0.500    --        --    Degenerate
Combined (TCN, 58-dim)               0.543    --        --    Degenerate
Oracle 10% (TCN, 7-dim)              0.783    --        --    Real signal
Oracle 50% (TCN, 7-dim)              0.811    --        --    Real signal
Oracle 90% (TCN, 7-dim)              0.933    --        --    Real signal
ESM-2 LR (1280-dim)                  0.541   0.495     0.597  Near-chance
ESM-2 RF (1280-dim)                  0.513   0.525     0.551  Near-chance
ESM-2 MLP (1280-dim)                 0.529   0.491     0.645  Near-chance
ESMDance LR (480-dim)                0.494   0.523     0.598  Near-chance
ESMDance RF (480-dim)                0.523   0.497     0.570  Near-chance
ESMDance MLP (480-

## Step 5: Notes and Observations
Record any unexpected findings, degenerate prediction behavior, or follow-up actions here as experiments complete.

- 